# Hands-on TrimNN tutorial

This notebook walks through the main steps in [`yuyang-0825/TrimNN`](https://github.com/yuyang-0825/TrimNN): preparing spatial omics coordinates, building the cellular community graph, generating an optional query motif, running the three TrimNN analysis modes, visualizing small motifs, and comparing against VF2 enumeration.

TrimNN identifies conserved cellular community motifs from spatial transcriptomics or proteomics data. The upstream workflow expects a CSV with `X`, `Y`, and `cell_type` columns, converts it to a Delaunay-triangulated GML graph, and then uses pretrained neural models to estimate motif occurrence or overrepresentation.

## Learning goals

By the end of this notebook you should be able to:

1. Set up the TrimNN repository and runtime assumptions.
2. Inspect the demo spatial cell table.
3. Convert cell coordinates and cell types into TrimNN GML input.
4. Generate a motif GML file for subgraph matching.
5. Run TrimNN Function 1: subgraph matching.
6. Run TrimNN Function 2: identify top overrepresented motifs of one size.
7. Run TrimNN Function 3: identify top overrepresented motifs across sizes.
8. Visualize size-1 to size-3 motifs on the original spatial coordinates.
9. Optionally compare Function 2 with VF2 enumeration.

> Practical note: the neural model steps can take minutes on a GPU and require the dependencies from the TrimNN repository. The notebook keeps those cells explicit and configurable.

## 0. Configure paths and execution mode

Run this first. Set `RUN_TRIMNN_COMMANDS = True` only after your environment has PyTorch, DGL, and the rest of TrimNN's dependencies installed.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

WORKDIR = Path.cwd()
REPO_URL = "https://github.com/yuyang-0825/TrimNN.git"
REPO_DIR = WORKDIR / "TrimNN"

# Keep False for teaching/demo mode. Switch to True to execute TrimNN CLI commands.
RUN_TRIMNN_COMMANDS = False

def run_command(cmd, cwd=REPO_DIR, run=RUN_TRIMNN_COMMANDS, check=True):
    """Print a shell command, and optionally run it."""
    printable = " ".join(shlex.quote(str(part)) for part in cmd)
    print(f"$ cd {cwd}\n$ {printable}")
    if not run:
        print("Skipped because RUN_TRIMNN_COMMANDS is False.")
        return None
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    return result

print(f"Notebook workspace: {WORKDIR}")
print(f"TrimNN checkout path: {REPO_DIR}")

## 1. Get TrimNN

Clone the upstream repository once. If you already cloned it, this cell leaves it untouched.

In [ ]:
if REPO_DIR.exists():
    print(f"Found existing repository: {REPO_DIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

print("Repository files:")
for path in sorted(REPO_DIR.iterdir()):
    print("-", path.name)

## 2. Install dependencies

The TrimNN README was tested with Python 3.9 and provides separate PyTorch/DGL install commands for CUDA and CPU. Choose the block matching your machine before running the rest of the notebook.

Recommended shell setup outside the notebook:

```bash
conda create -n TrimNNEnv python=3.9
conda activate TrimNNEnv

# CPU-only example from the TrimNN README
conda install pytorch==1.13.1 torchvision==0.14.1 torchaudio==0.13.1 cpuonly -c pytorch
pip install dgl==1.1.2 -f https://data.dgl.ai/wheels/repo.html
pip install -r requirements.txt
```

If you are on a CUDA Linux machine, use the CUDA-specific PyTorch and DGL commands from the upstream README instead.

In [ ]:
# Lightweight environment check. This does not install packages.
packages = ["torch", "dgl", "networkx", "pandas", "scipy", "sklearn", "igraph"]
for package in packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "installed")
        print(f"{package}: {version}")
    except Exception as exc:
        print(f"{package}: not available ({exc.__class__.__name__})")

## 3. Inspect input spatial omics data

TrimNN needs at least three columns:

- `X`: cell x-coordinate
- `Y`: cell y-coordinate
- `cell_type`: cell type label

The demo file is `demo_data/demo_data.csv`.

In [ ]:
import pandas as pd

demo_csv = REPO_DIR / "demo_data" / "demo_data.csv"
df = pd.read_csv(demo_csv)
print(df.shape)
display(df.head())
display(df["cell_type"].value_counts().rename_axis("cell_type").reset_index(name="n_cells"))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 6))
for cell_type, group in df.groupby("cell_type"):
    ax.scatter(group["X"], group["Y"], s=8, alpha=0.8, label=cell_type)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Demo spatial cells by cell type")
ax.legend(markerscale=2, bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()

## 4. Convert CSV to cellular community GML

`csv2gml.py` applies Delaunay triangulation to the cell coordinates and writes a graph in GML format. The optional `-prune True` mode removes outlier edges longer than the 99th-percentile edge length, which can help reduce boundary artifacts.

In [ ]:
target_gml = REPO_DIR / "demo_data" / "demo_data.gml"

run_command([
    sys.executable, "csv2gml.py",
    "-target", "demo_data/demo_data.csv",
    "-out", "demo_data/demo_data.gml",
    "-prune", "False",
])

In [ ]:
# Optional: inspect the generated mapping between original cell types and integer labels.
mapping_csv = REPO_DIR / "demo_data" / "cell_type_to_id.csv"
if mapping_csv.exists():
    display(pd.read_csv(mapping_csv))
else:
    print("Run the csv2gml step to generate cell_type_to_id.csv")

## 5. Generate a query motif GML for subgraph matching

For Function 1, TrimNN needs a target graph and a motif graph. The repository can generate a simple motif GML from a label string such as `Micro_Micro_Micro`. Use labels that actually occur in your dataset.

In [ ]:
motif_size = 3
motif_label = "Micro_Micro_Micro"

run_command([
    sys.executable, "csv2gml.py",
    "-target", "demo_data/demo_data.csv",
    "-out", "demo_data/demo_data.gml",
    "-motif_size", str(motif_size),
    "-motif_label", motif_label,
    "-prune", "False",
])

motif_gml = REPO_DIR / "demo_data" / f"size-{motif_size}.gml"
print(f"Expected motif file: {motif_gml}")

## 6. Function 1: subgraph matching

This mode predicts the number of occurrences of one input cellular community motif in the target cellular community graph.

Key arguments:

- `-function subgraph_matching`
- `-motif`: query motif GML
- `-target`: target cellular community GML
- `-k`: k-hop enclosed graph around each node; the README example uses `2`
- `-outpath`: output directory

In [ ]:
run_command([
    sys.executable, "TrimNN.py",
    "-function", "subgraph_matching",
    "-motif", "demo_data/size-3.gml",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-outpath", "result_function1/",
])

In [ ]:
result_dir = REPO_DIR / "result_function1"
if result_dir.exists():
    for path in sorted(result_dir.iterdir()):
        print(path.name)
        if path.suffix.lower() in {".csv", ".txt"}:
            print(path.read_text()[:1000])
else:
    print("Run Function 1 to create result_function1/")

## 7. Function 2: top overrepresented motifs of a specific size

This mode searches motifs of one size and reports predicted occurrence counts. The upstream demo uses size 3 and 8 cell types.

In [ ]:
n_cell_types = df["cell_type"].nunique()
print(f"Number of cell types in demo CSV: {n_cell_types}")

run_command([
    sys.executable, "TrimNN.py",
    "-function", "specific_size",
    "-size", "3",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-celltype", str(n_cell_types),
    "-outpath", "result_function2/",
])

In [ ]:
result_dir = REPO_DIR / "result_function2"
if result_dir.exists():
    print("Function 2 outputs:")
    for path in sorted(result_dir.iterdir()):
        print("-", path.name)
    csv_outputs = sorted(result_dir.glob("*.csv"))
    if csv_outputs:
        display(pd.read_csv(csv_outputs[0]).head(10))
else:
    print("Run Function 2 to create result_function2/")

## 8. Function 3: top overrepresented motifs across sizes

This mode grows motifs from size 3 up to the requested maximum size. The README example uses `-size 4` and greedy search.

In [ ]:
run_command([
    sys.executable, "TrimNN.py",
    "-function", "all_size",
    "-size", "4",
    "-k", "2",
    "-target", "demo_data/demo_data.gml",
    "-celltype", str(n_cell_types),
    "-outpath", "result_function3/",
    "-search", "greedy",
])

In [ ]:
result_dir = REPO_DIR / "result_function3"
if result_dir.exists():
    print("Function 3 outputs:")
    for path in sorted(result_dir.iterdir()):
        print("-", path.name)
else:
    print("Run Function 3 to create result_function3/")

## 9. Visualize a motif on the spatial cell map

`visualize.py` supports motifs of size 1 to 3. For larger motifs, the repository suggests custom visualization because structural diversity increases quickly.

In [ ]:
visual_motif_size = 3
visual_motif_label = "CTX-Ex_CTX-Ex_CTX-Ex"

run_command([
    sys.executable, "visualize.py",
    "-target", "demo_data/demo_data.csv",
    "-outpath", "visualization/",
    "-motif_size", str(visual_motif_size),
    "-motif_label", visual_motif_label,
])

In [ ]:
from IPython.display import Image, display

viz_dir = REPO_DIR / "visualization"
if viz_dir.exists():
    for path in sorted(viz_dir.iterdir()):
        print("-", path.name)
        if path.suffix.lower() in {".png", ".jpg", ".jpeg"}:
            display(Image(filename=str(path)))
else:
    print("Run the visualization step to create visualization/")

## 10. Optional VF2 enumeration comparison

The repository includes `vf2_analysis.py` as a classical enumeration baseline for Function 2. This can be useful for small motifs, but enumeration becomes expensive as motif size increases.

In [ ]:
run_command([
    sys.executable, "vf2_analysis.py",
    "-size", "3",
    "-target", "demo_data/demo_data.gml",
    "-celltype", str(n_cell_types),
    "-outpath", "result_vf2_function2/",
])

## 11. Use your own spatial omics dataset

To adapt this notebook:

1. Put your CSV somewhere accessible from the notebook.
2. Confirm it has `X`, `Y`, and `cell_type` columns.
3. Run `csv2gml.py` with your CSV as `-target`.
4. Set `-celltype` to the number of unique cell types in your data.
5. Choose motif labels by joining cell type names with underscores, for example `Tcell_Macrophage_Fibroblast`.

The cell below is a template. Edit `my_csv` and `my_out_gml` before running.

In [ ]:
my_csv = Path("/path/to/my_spatial_cells.csv")
my_out_gml = Path("demo_data/my_spatial_cells.gml")

if my_csv.exists():
    my_df = pd.read_csv(my_csv)
    required = {"X", "Y", "cell_type"}
    missing = required.difference(my_df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    print(my_df.shape)
    print(f"Unique cell types: {my_df['cell_type'].nunique()}")
    run_command([
        sys.executable, "csv2gml.py",
        "-target", str(my_csv),
        "-out", str(my_out_gml),
        "-prune", "True",
    ])
else:
    print("Edit my_csv to point to your data before running this template.")

## 12. Troubleshooting checklist

- `undefined symbol: iJIT_NotifyEvent`: the upstream README suggests installing `mkl==2024.0`.
- `ModuleNotFoundError: dgl`: install the DGL build matching your PyTorch/CUDA setup.
- No `cell_type_to_id.csv`: rerun `csv2gml.py` and check that the output folder is writable.
- Unexpected cell type count: inspect spelling and whitespace in `cell_type` values before running `TrimNN.py`.
- Very long runtime: start with size 3 and the demo data, then increase motif size only after the pipeline works.